## Baseline scores on semart using CLIP

In [13]:
%load_ext autoreload
%autoreload 2

import os
import json
import torch
import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd


from typing import List
from tqdm import tqdm 
from src.model import SheafMultimodalGNN
from src.utils import *
from src.data import *
from src.ClusterData import ClusterData, ClusterLoader
from src.metrics import *

triplets = '../artistic_sheaf/data/triplets_semart_test_csv.json'
#triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 9914 triplets from ../artistic_sheaf/data/triplets_semart_test_csv.json


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=49-val_loss=5.17.ckpt", map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()
print()

Using device: mps



In [5]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data#.to(device)


100%|██████████| 9914/9914 [00:19<00:00, 500.80it/s]


In [6]:
print(test_graph_data.edge_index.shape[1])

9914


In [6]:
# print("Loaded test data with {} nodes.".format(len(test_node_to_id.keys())))
# print("Creating data loaders...")
# #batch_size = 768
# print("Number of parts:", 1)
# test_graph_data.num_nodes = len(test_graph_data.x)
# #test_graph_data.orig_id = torch.arange(test_graph_data.edge_index.shape[1])
# #test_graph_data.edge_attr = torch.cat([test_graph_data.edge_attr, test_graph_data.orig_id.unsqueeze(1)], dim=1)

# dataset = ClusterData(test_graph_data, num_parts=1, recursive=False, save_dir='data/clusters_test')
# test_loader = ClusterLoader(dataset, batch_size=1, shuffle=False)
    


In [7]:
#clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
with torch.no_grad():
    x_img, x_text, edge_index, edge_attr = process_batch(test_graph_data, 'test')
    x_img = x_img.to(device)
    x_text = x_text.to(device)
    edge_index = edge_index.to(device)
    edge_attr = edge_attr.to(device)
    print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
    
    embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
    clip_images = F.normalize(embeddings[: len(edge_attr), :], dim=1)
    clip_texts = F.normalize(embeddings[len(edge_attr):, :], dim=1)

print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")


torch.Size([1570, 3, 224, 224]) torch.Size([4261, 77]) torch.Size([2, 9914]) torch.Size([9914, 77])
Checking maps tensor([[-0.1383],
        [ 0.7925],
        [ 0.9286],
        [ 0.5481],
        [ 0.3839]], device='mps:0')
Checking maps tensor([[-0.0580],
        [ 0.6552],
        [ 0.7414],
        [ 0.4396],
        [ 0.2873]], device='mps:0')
Checking maps tensor([[0.0756],
        [0.7970],
        [0.8164],
        [0.6043],
        [0.4229]], device='mps:0')
Extracted 9914 text embeddings, each of shape torch.Size([512])


In [8]:
from src.metrics import *
metrics_i2t = compute_clip_metrics(clip_images, clip_texts)
metrics_t2i = compute_clip_metrics(clip_texts, clip_images)
        
print(metrics_i2t)
print(metrics_t2i)

/Users/ludovicaschaerf/Desktop/Sheaf_Art/artistic_sheaf/src/metrics.py:70: UserWarning: MPS: nonzero op is not natively supported for the provided input on MacOS14Falling back on CPU. This may have performance implications.See github.com/pytorch/pytorch/issues/122916 for further info (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/mps/operations/Indexing.mm:352.)
  match_ranks = (rankings == ground_truth).nonzero(as_tuple=False)[:, 1]  # rank index where correct match appears


{'Recall@1': 0.005345975514501333, 'Recall@5': 0.026225538924336433, 'Recall@10': 0.04740770533680916, 'Mean Rank': 172.8137969970703, 'Median Rank': 145}
{'Recall@1': 0.007968529127538204, 'Recall@5': 0.033286262303590775, 'Recall@10': 0.057998787611722946, 'Mean Rank': 169.04185485839844, 'Median Rank': 143}


In [ ]:
# preds_img = torch.empty((len(loaded_data), 512))
# preds_txt = torch.empty((len(loaded_data), 512))
    
# with torch.no_grad():
#     for batch in tqdm(test_loader):
#         batch = batch.to(device)
#         batch.x, batch.edge_index, batch.edge_attr = batch.x, batch.edge_index, batch.edge_attr
#         batch.orig_id = batch.edge_attr[:, -1]
#         batch.edge_attr = batch.edge_attr[:, :-1]
#         img_emb, txt_emb, orig_ids = model.step(batch, 0, split='predict')
#         orig = orig_ids.cpu().numpy()
#         preds_img[orig] = img_emb.detach().cpu()
#         preds_txt[orig] = txt_emb.detach().cpu()
        
# clip_images = F.normalize(preds_img, dim=1)
# clip_texts = F.normalize(preds_txt, dim=1)

In [9]:
clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

In [10]:
# save embeddings images and test

np.save('data/clip_images_train.npy', clip_images)
np.save('data/clip_texts_train.npy', clip_texts)


In [7]:
clip_images = np.load('data/clip_images_full.npy')
clip_texts = np.load('data/clip_texts_full.npy')

In [8]:
# take a subset of the image embeddings and plot them with plotly interactively (in 2D using umap) showing the edge_index[0, i] on hover
import umap
import plotly.express as px
reducer = umap.UMAP()
#from sklearn.decomposition import PCA
#reducer = PCA(n_components=2)

embedding_2d = reducer.fit_transform(np.concatenate([clip_images[: 400], clip_texts[: 400]], axis=0) ) # take only first 
fig = px.scatter(x=embedding_2d[:, 0], y=embedding_2d[:, 1],
                hover_data=[np.concatenate([np.arange(400), np.arange(400)], axis=0),
                            np.concatenate([test_graph_data.edge_index[0, :400].cpu().numpy(), test_graph_data.edge_index[1, :400].cpu().numpy()], axis=0)], 
                color=['images']*400 + ['text']*400)
fig.show()

In [13]:
# from collections import defaultdict 
# def reorder_predictions_by_link_item(
#     predictions: np.ndarray,
#     new_list: str,   # "data/full_triplets.json" (the list used to produce predictions)
#     old_triplets_path: str,   # the new file with same links but different item2 assignments
#     item = 'item2'  # which item to use for matching (default 'item2' for text predictions)
# ):
#     """
#     Reorder the text predictions to match the order of (link, item2) in the new triplet file.
#     Assumes:
#       - predictions_txt[i] corresponds to old_list[i]['item2'] with old_list[i]['link'].
#       - Keys used for matching are (link, item2).
#       - Handles duplicate (link, item2) by consuming old indices FIFO.
#     """
#     # Load lists
#     old_list = load_json_data(old_triplets_path)

#     # Build mapping: (link, item2) -> queue of old indices
#     pos_by_key = defaultdict(list)
#     for idx, tr in enumerate(old_list):
#         link = tr.get("link")
#         item2 = tr.get(item)
#         pos_by_key[(link, item2)].append(idx)

#     print(len(pos_by_key), "unique (link,item) pairs in the old file.")
#     # Build reorder indices to match new_list order
#     reorder_indices = []
#     missing = []
#     for tr in new_list:
#         key = (tr.get("link"), tr.get(item))
#         if pos_by_key[key]:
#             reorder_indices.append(pos_by_key[key].pop(0))  # consume one occurrence
#         else:
#             missing.append(key)

#     if missing:
#         # Raise for visibility; switch to a warning if partial overlap is expected.
#         example = missing[:5]
#         print(
#             f"{len(missing)} (link,{item}) pairs in the new file were not found in the old predictions. "
#             f"Examples: {example}"
#         )

#     # Reorder predictions
#     idx_t = np.array(reorder_indices)
#     predictions_reordered = predictions[idx_t]
#     return predictions_reordered


In [14]:
# predictions_txt_new_order = reorder_predictions_by_link_item(
#     clip_texts,
#     new_list=loaded_data[len(loaded_data)//2:],  # use only the test portion of the loaded data
#     old_triplets_path="data/triplets_semart_test_csv.json",
#     item='item2'
# )
# print(predictions_txt_new_order.shape)
    

In [15]:
# predictions_img_new_order = reorder_predictions_by_link_item(
#     clip_images,
#     new_list=loaded_data[:len(loaded_data)//2],  # use only the test portion of the loaded data
#     old_triplets_path="data/triplets_semart_test_csv.json",
#     item='item1'
# )
# print(predictions_img_new_order.shape)


In [16]:
# clip_texts = predictions_txt_new_order

# clip_images = predictions_img_new_order

### Image-to-text retrieval	
### Text-to-image retrieval		
r@1	r@5	r@10	

In [14]:
loaded_data = load_json_data("data/triplets_semart_test_csv.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_csv.json")

Loaded 9914 triplets from data/triplets_semart_test_csv.json


In [15]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data)
print(f"Adjacency matrix shape: {adj_matrix.shape}")

Adjacency matrix shape: (1069, 5153)


In [16]:
sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data], 
                            [t["item2"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Similarity matrix shape: (1069, 5153)


In [17]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

{'t2i_precision@1': tensor(0.0047),
 't2i_recall@1': tensor(0.0006),
 't2i_ndcg@1': tensor(0.0047),
 't2i_precision@5': tensor(0.0060),
 't2i_recall@5': tensor(0.0034),
 't2i_ndcg@5': tensor(0.0058),
 't2i_precision@10': tensor(0.0047),
 't2i_recall@10': tensor(0.0052),
 't2i_ndcg@10': tensor(0.0056),
 'i2t_precision@1': tensor(0.0027),
 'i2t_recall@1': tensor(0.0021),
 'i2t_ndcg@1': tensor(0.0027),
 'i2t_precision@5': tensor(0.0037),
 'i2t_recall@5': tensor(0.0127),
 'i2t_ndcg@5': tensor(0.0083),
 'i2t_precision@10': tensor(0.0035),
 'i2t_recall@10': tensor(0.0253),
 'i2t_ndcg@10': tensor(0.0122),
 'mean_precision@1': tensor(0.0037),
 'mean_recall@1': tensor(0.0014),
 'mean_ndcg@1': tensor(0.0037),
 'mean_precision@5': tensor(0.0048),
 'mean_recall@5': tensor(0.0080),
 'mean_ndcg@5': tensor(0.0070),
 'mean_precision@10': tensor(0.0041),
 'mean_recall@10': tensor(0.0153),
 'mean_ndcg@10': tensor(0.0089)}

### Retrieval per type of relationship

In [18]:
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    #print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    #print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t["item2"] for t in loaded_data_new], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
    #print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))
    # Print which query gives which recommendation (text or image path)
    # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
    recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=5)

    query_field = 'item1'  # image path
    rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
    idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

    for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
        query = loaded_data_new[i][query_field]
        recommendations = [idx_to_txt[j] for j in rec_indices]
        print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
        print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
        print("Recommendations:")
        for rec in recommendations:
            print(f"  - {rec}")
        print("-" * 40)

Processing type: context
{'t2i_precision@1': tensor(0.0442), 't2i_recall@1': tensor(0.0267), 't2i_ndcg@1': tensor(0.0442), 't2i_precision@5': tensor(0.0284), 't2i_recall@5': tensor(0.0871), 't2i_ndcg@5': tensor(0.0636), 't2i_precision@10': tensor(0.0304), 't2i_recall@10': tensor(0.1784), 't2i_ndcg@10': tensor(0.0964), 'i2t_precision@1': tensor(0.0380), 'i2t_recall@1': tensor(0.0377), 'i2t_ndcg@1': tensor(0.0380), 'i2t_precision@5': tensor(0.0290), 'i2t_recall@5': tensor(0.1448), 'i2t_ndcg@5': tensor(0.0900), 'i2t_precision@10': tensor(0.0264), 'i2t_recall@10': tensor(0.2636), 'i2t_ndcg@10': tensor(0.1283), 'mean_precision@1': tensor(0.0411), 'mean_recall@1': tensor(0.0322), 'mean_ndcg@1': tensor(0.0411), 'mean_precision@5': tensor(0.0287), 'mean_recall@5': tensor(0.1159), 'mean_ndcg@5': tensor(0.0768), 'mean_precision@10': tensor(0.0284), 'mean_recall@10': tensor(0.2210), 'mean_ndcg@10': tensor(0.1124)}
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/41294-10ladisl.jpg
Gr

## Zero shot classification

In [5]:
annotations = '../SemArt/semart_test.csv'
df = pd.read_csv(annotations, sep='\t', encoding='latin1')
df.head(), df.shape

(           IMAGE_FILE                                        DESCRIPTION  \
 0  41294-10ladisl.jpg  Of the Hungarian kings St Ladislas is perhaps ...   
 1   42791-1sacris.jpg  This ceiling painting in the sacristy of San S...   
 2   14376-worship.jpg  In the same period when the most talented arti...   
 3  24776-annuncia.jpg  Based on its style the Annunciation is attribu...   
 4  23845-3manet04.jpg  The 1870s were rich in female models for Manet...   
 
                       AUTHOR                             TITLE  \
 0  UNKNOWN MASTER, Hungarian  Saint Ladislaus, King of Hungary   
 1            VERONESE, Paolo          Coronation of the Virgin   
 2         FRANCKEN, Frans II        Worship of the Golden Calf   
 3         MASTER of Flémalle                      Annunciation   
 4             MANET, Edouard        Brunette with Bare Breasts   
 
                       TECHNIQUE     DATE       TYPE     SCHOOL  TIMEFRAME  
 0   Oil on wood, 103 x 101,3 cm  c. 1600  religious  H

In [6]:
loaded_data_new = []
for itm in df['IMAGE_FILE']:
    loaded_data_new.append({})
    loaded_data_new[-1]['item1'] = 'Images/' + itm
    author = df[df['IMAGE_FILE'] == itm]['AUTHOR'].values[0]
    loaded_data_new[-1]['author'] = f"Artwork by {author}"
    timeframe = df[df['IMAGE_FILE'] == itm]['TIMEFRAME'].values[0]
    loaded_data_new[-1]['timeframe'] = f"Artwork painted in {timeframe}"
    school = df[df['IMAGE_FILE'] == itm]['SCHOOL'].values[0]
    loaded_data_new[-1]['school'] = f"Artwork from the {school} school"
    material = df[df['IMAGE_FILE'] == itm]['TECHNIQUE'].values[0].split(',')[0]
    loaded_data_new[-1]['material'] = f"Artwork made with {material}"
    genre = df[df['IMAGE_FILE'] == itm]['TYPE'].values[0]
    loaded_data_new[-1]['genre'] = f"Artwork of the {genre} genre"
    loaded_data_new[-1]['link'] = 'metadata'

print(f"Updated loaded_data with authors, total items: {len(loaded_data_new)}")

Updated loaded_data with authors, total items: 1069


In [7]:
loaded_data_new[0].keys()

dict_keys(['item1', 'author', 'timeframe', 'school', 'material', 'genre', 'link'])

In [8]:
item2 = 'school' #author, timeframe, school, material, genre

In [9]:
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_new, preprocess, tokenizer, 
                                                                           base_folder='../SemArt/', item2=item2)
test_graph_data = test_graph_data.to(device)
print("Loaded test data with {} nodes.".format(len(test_node_to_id.keys())))
print("Creating data loaders...")
test_dataset = GraphEdgeDataset(test_graph_data)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
for batch in test_loader:
    x_img, x_text, edge_index, edge_attr = process_batch(batch, 'test')
    embeddings = model(x_img, x_text, edge_index, edge_attr)
    clip_images_cls = F.normalize(embeddings[edge_index[0, :]], dim=1).cpu().detach().numpy()
    clip_texts_author = F.normalize(embeddings[edge_index[1, :]], dim=1).cpu().detach().numpy()

    print(f"Extracted {len(clip_texts_author)} text embeddings, each of shape {clip_texts_author[0].shape}")


100%|██████████| 1069/1069 [00:10<00:00, 104.70it/s]


Loaded test data with 1092 nodes.
Creating data loaders...


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


tensor(0, device='mps:0') tensor(1091, device='mps:0')
Extracted 1069 text embeddings, each of shape (512,)


In [10]:
adj_matrix_author, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new, field=item2)
sim_matrix_author = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t[item2] for t in loaded_data_new], 
                            clip_images_cls, clip_texts_author,
                            img_to_idx, txt_to_idx)

print(f"Adjacency matrix shape: {adj_matrix_author.shape}", 
      f"Similarity matrix shape: {sim_matrix_author.shape}")

Reordered image embeddings shape: (1069, 512)
Reordered text embeddings shape: (23, 512)
Adjacency matrix shape: (1069, 23) Similarity matrix shape: (1069, 23)


In [23]:
adj_matrix_author[10]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0])

In [24]:
sim_matrix_author[10].argmax()

np.int64(13)

In [11]:
compute_image_to_text_accuracy(sim_matrix_author, adj_matrix_author)

1.0

In [29]:
txt_to_idx

{'Artwork from the American school': 0,
 'Artwork from the Austrian school': 1,
 'Artwork from the Belgian school': 2,
 'Artwork from the Bohemian school': 3,
 'Artwork from the Danish school': 4,
 'Artwork from the Dutch school': 5,
 'Artwork from the English school': 6,
 'Artwork from the Flemish school': 7,
 'Artwork from the French school': 8,
 'Artwork from the German school': 9,
 'Artwork from the Greek school': 10,
 'Artwork from the Hungarian school': 11,
 'Artwork from the Irish school': 12,
 'Artwork from the Italian school': 13,
 'Artwork from the Netherlandish school': 14,
 'Artwork from the Other school': 15,
 'Artwork from the Polish school': 16,
 'Artwork from the Portuguese school': 17,
 'Artwork from the Russian school': 18,
 'Artwork from the Scottish school': 19,
 'Artwork from the Spanish school': 20,
 'Artwork from the Swedish school': 21,
 'Artwork from the Swiss school': 22}

In [ ]:
# Print which query gives which recommendation (text or image path)
# For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
recs = get_top_k_recommendations(torch.Tensor(sim_matrix_author), k=5)

query_field = 'item1'  # image path
rec_field = item2      # e.g., 'timeframe', 'author', etc.
idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

for i, rec_indices in enumerate(recs):
    query = loaded_data_new[i][query_field]
    recommendations = [idx_to_txt[j] for j in rec_indices]
    print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
    print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
    print("Recommendations:")
    for rec in recommendations:
        print(f"  - {rec}")
    print("-" * 40)

Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/41294-10ladisl.jpg
Ground Truth: Artwork from the Hungarian school
Recommendations:
  - Artwork from the Dutch school
  - Artwork from the Belgian school
  - Artwork from the German school
  - Artwork from the Danish school
  - Artwork from the Austrian school
----------------------------------------
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/42791-1sacris.jpg
Ground Truth: Artwork from the Italian school
Recommendations:
  - Artwork from the Dutch school
  - Artwork from the Belgian school
  - Artwork from the German school
  - Artwork from the Danish school
  - Artwork from the Austrian school
----------------------------------------
Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/Images/14376-worship.jpg
Ground Truth: Artwork from the Flemish school
Recommendations:
  - Artwork from the Netherlandish school
  - Artwork from the Flemish school
  - Artwork from the Dutch school
  - Artwork from the